# Silver Layer - Data Transformation

Cleanses and enriches bronze data, performs joins, and applies business logic.

**Source:** workspace.bronze_chocolate  
**Target:** workspace.silver_chocolate

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, trim, upper, lower, regexp_replace, when, coalesce,
    to_date, to_timestamp, year, month, dayofmonth, quarter,
    current_timestamp, lit, count, sum as spark_sum, avg, round as spark_round,
    row_number, dense_rank, lag, lead
)
from pyspark.sql.window import Window
from pyspark.sql.types import *

# Configuration
BRONZE_CATALOG = "workspace"
BRONZE_SCHEMA = "bronze_chocolate"
SILVER_CATALOG = "workspace"
SILVER_SCHEMA = "silver_chocolate"



✓ Imports loaded
✓ Source: workspace.bronze_chocolate
✓ Target: workspace.silver_chocolate


In [0]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SILVER_CATALOG}.{SILVER_SCHEMA}")

✓ Schema workspace.silver_chocolate ready


In [0]:
bronze_tables_df = spark.sql(f"SHOW TABLES IN {BRONZE_CATALOG}.{BRONZE_SCHEMA}")
bronze_tables = [row.tableName for row in bronze_tables_df.collect() if row.tableName.startswith('bronze_')]


✓ Found 5 Bronze table(s):
  • bronze_calendar: 731 rows
  • bronze_customers: 50,000 rows
  • bronze_products: 200 rows
  • bronze_sales: 1,000,000 rows
  • bronze_stores: 100 rows


In [0]:


# Read Bronze tables
df_sales_bronze = spark.table(f"{BRONZE_CATALOG}.{BRONZE_SCHEMA}.bronze_sales")
df_products_bronze = spark.table(f"{BRONZE_CATALOG}.{BRONZE_SCHEMA}.bronze_products")
df_stores_bronze = spark.table(f"{BRONZE_CATALOG}.{BRONZE_SCHEMA}.bronze_stores")
df_customers_bronze = spark.table(f"{BRONZE_CATALOG}.{BRONZE_SCHEMA}.bronze_customers")




df_sales_clean = df_sales_bronze \
    .dropDuplicates() \
    .na.drop(subset=["order_id", "order_date", "product_id"])


df_sales_clean = df_sales_clean.withColumn(
    "sale_date",
    to_date(col("order_date"), "yyyy-MM-dd")
)


df_sales_clean = df_sales_clean \
    .withColumn("quantity_clean", coalesce(col("quantity").cast("int"), lit(0))) \
    .withColumn("unit_price_clean", coalesce(col("unit_price").cast("double"), lit(0.0))) \
    .withColumn("discount_clean", coalesce(col("discount").cast("double"), lit(0.0))) \
    .withColumn("revenue_clean", coalesce(col("revenue").cast("double"), lit(0.0))) \
    .withColumn("cost_clean", coalesce(col("cost").cast("double"), lit(0.0))) \
    .withColumn("profit_clean", coalesce(col("profit").cast("double"), lit(0.0)))


df_sales_enriched = df_sales_clean \
    .join(df_products_bronze.select("product_id", "product_name", "brand", "category"), "product_id", "left") \
    .join(df_stores_bronze.select("store_id", "store_name", "city", "country", "store_type"), "store_id", "left") \
    .join(df_customers_bronze.select("customer_id", "age", "gender", "loyalty_member"), "customer_id", "left")


df_sales_enriched = df_sales_enriched \
    .withColumn("profit_margin_pct", 
                when(col("revenue_clean") > 0, (col("profit_clean") / col("revenue_clean")) * 100).otherwise(0)) \
    .withColumn("discount_amount", col("quantity_clean") * col("unit_price_clean") * col("discount_clean"))


df_sales_enriched = df_sales_enriched \
    .withColumn("year", year(col("sale_date"))) \
    .withColumn("quarter", quarter(col("sale_date"))) \
    .withColumn("month", month(col("sale_date"))) \
    .withColumn("day", dayofmonth(col("sale_date")))


df_sales_enriched = df_sales_enriched \
    .withColumn("silver_processed_timestamp", current_timestamp()) \
    .withColumn("is_valid", lit(True))


df_sales_silver = df_sales_enriched.select(

    "order_id", "product_id", "store_id", "customer_id",

    "sale_date", "year", "quarter", "month", "day",

    "product_name", "brand", "category",

    "store_name", "city", "country", "store_type",

    "age", "gender", "loyalty_member",

    "quantity_clean", "unit_price_clean", "discount_clean", "discount_amount",
    "revenue_clean", "cost_clean", "profit_clean", "profit_margin_pct",

    "is_valid", "silver_processed_timestamp",
    "bronze_ingestion_timestamp", "bronze_source_file"
)




silver_sales_path = f"{SILVER_CATALOG}.{SILVER_SCHEMA}.silver_sales"
df_sales_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(silver_sales_path)

print(f"Silver sales: {df_sales_silver.count():,} rows -> {silver_sales_path}")


📊 Processing Sales data...
  • Bronze sales rows: 1,000,000
  • Columns: ['order_id', 'order_date', 'product_id', 'store_id', 'customer_id', 'quantity', 'unit_price', 'discount', 'revenue', 'cost', 'profit', 'bronze_ingestion_timestamp', 'bronze_source_file']
  • After cleansing: 1,000,000 rows
  • Removed duplicates: 0 rows
  ✓ Saved to: workspace.silver_chocolate.silver_sales


In [0]:


df_products = df_sales_silver \
    .select("product_id", "product_name", "brand", "category") \
    .distinct() \
    .withColumn("product_name_clean", trim(col("product_name"))) \
    .withColumn("brand_clean", trim(col("brand"))) \
    .withColumn("category_clean", trim(col("category"))) \
    .withColumn("silver_processed_timestamp", current_timestamp())

df_products_final = df_products.select(
    "product_id", "product_name", "product_name_clean", "brand", "brand_clean", "category", "category_clean",
    "silver_processed_timestamp"
)



silver_products_path = f"{SILVER_CATALOG}.{SILVER_SCHEMA}.silver_products"
df_products_final.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(silver_products_path)

print(f"Silver products: {df_products_final.count():,} rows -> {silver_products_path}")


🍫 Creating Product Dimension...
  • Unique products: 202
  ✓ Saved to: workspace.silver_chocolate.silver_products


In [0]:


df_stores = df_sales_silver \
    .select("store_id", "store_name", "city", "country", "store_type") \
    .distinct() \
    .withColumn("store_name_clean", trim(col("store_name"))) \
    .withColumn("city_clean", trim(col("city"))) \
    .withColumn("country_clean", trim(upper(col("country")))) \
    .withColumn("store_type_clean", trim(col("store_type"))) \
    .withColumn("silver_processed_timestamp", current_timestamp())

df_stores_final = df_stores.select(
    "store_id", "store_name", "store_name_clean", "city", "city_clean", 
    "country", "country_clean", "store_type", "store_type_clean",
    "silver_processed_timestamp"
)



silver_stores_path = f"{SILVER_CATALOG}.{SILVER_SCHEMA}.silver_stores"
df_stores_final.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(silver_stores_path)

print(f"Silver stores: {df_stores_final.count():,} rows -> {silver_stores_path}")


🏪 Creating Store Dimension...
  • Unique stores: 100
  ✓ Saved to: workspace.silver_chocolate.silver_stores


In [0]:


df_geography = df_sales_silver \
    .select("country") \
    .distinct() \
    .withColumn("country_id", row_number().over(Window.orderBy("country"))) \
    .withColumn("country_name", trim(upper(col("country")))) \
    .withColumn("silver_processed_timestamp", current_timestamp())

df_geography_final = df_geography.select(
    "country_id", "country", "country_name",
    "silver_processed_timestamp"
)



silver_geography_path = f"{SILVER_CATALOG}.{SILVER_SCHEMA}.silver_geography"
df_geography_final.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(silver_geography_path)

print(f"Silver geography: {df_geography_final.count():,} rows -> {silver_geography_path}")


🌍 Creating Geography Dimension...


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


  • Unique countries: 6
  ✓ Saved to: workspace.silver_chocolate.silver_geography


In [0]:
print("\nSilver layer summary:")

silver_tables = [
    "silver_sales",
    "silver_products",
    "silver_stores",
    "silver_geography"
]

for table_name in silver_tables:
    table_path = f"{SILVER_CATALOG}.{SILVER_SCHEMA}.{table_name}"
    df = spark.table(table_path)
    row_count = df.count()
    col_count = len(df.columns)
    print(f"{table_path}: {row_count:,} rows, {col_count} columns")




SILVER LAYER TABLES
  • workspace.silver_chocolate.silver_sales: 1,000,000 rows, 31 columns
  • workspace.silver_chocolate.silver_products: 202 rows, 8 columns
  • workspace.silver_chocolate.silver_stores: 100 rows, 10 columns
  • workspace.silver_chocolate.silver_geography: 6 rows, 4 columns

✓ Silver tables are ready for Gold layer processing!
